In [0]:
%run ./connectionNotebook

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, row_number
from pyspark.sql.window import Window

bronze_path = "abfss://bronze@adlsg2rag.dfs.core.windows.net/sqlserver/orders/load_date=2026-03-13/"

In [0]:
df = spark.read.format("parquet").load(bronze_path)

df.printSchema()

In [0]:
# PK validation
df_clean = df.filter(col("OrderID").isNotNull() & (col("OrderID") != 0))

In [0]:
# Date validation
df_clean = df_clean.withColumn("OrderDate", to_date(col("OrderDate")))


df_clean = df_clean.filter(col("OrderDate").isNotNull())

In [0]:
# Processing timestamp
df_clean = df_clean.withColumn("processed_ts", current_timestamp())

In [0]:
# Duplicate check
w = Window.partitionBy("OrderID").orderBy(col("processed_ts").desc())
df_clean = (
    df_clean.withColumn("rn", row_number().over(w))
            .filter(col("rn") == 1)
            .drop("rn")
)

In [0]:
# Select and rename columns
df_clean = df_clean.select(
    col("OrderID").alias("src_OrderID"),
    col("CustomerID").alias("src_CustomerID"),
    col("OrderDate").alias("src_OrderDate"),
    col("OrderAmount").alias("src_OrderAmount"),
    col("processed_ts")
)

In [0]:
catalog_name = 'adbrag'
schema_name = 'silver'

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.orders_silver
(
    src_OrderID INT,
    src_CustomerID INT,
    src_OrderDate DATE,
    src_OrderAmount DECIMAL(10,2),
    processed_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_OrderID)
""")



In [0]:
df_clean.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{schema_name}.orders_silver"
)